<a href="https://colab.research.google.com/github/KAMAL0657/KAMAL-HUSSAIN/blob/main/ChurnPrediction_checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![COUR_IPO.png](attachment:COUR_IPO.png)

# Welcome to the Data Science Coding Challange!

Test your skills in a real-world coding challenge. Coding Challenges provide CS & DS Coding Competitions with Prizes and achievement badges!

CS & DS learners want to be challenged as a way to evaluate if they’re job ready. So, why not create fun challenges and give winners something truly valuable such as complimentary access to select Data Science courses, or the ability to receive an achievement badge on their Coursera Skills Profile - highlighting their performance to recruiters.

## Introduction

In this challenge, you'll get the opportunity to tackle one of the most industry-relevant maching learning problems with a unique dataset that will put your modeling skills to the test. Subscription services are leveraged by companies across many industries, from fitness to video streaming to retail. One of the primary objectives of companies with subscription services is to decrease churn and ensure that users are retained as subscribers. In order to do this efficiently and systematically, many companies employ machine learning to predict which users are at the highest risk of churn, so that proper interventions can be effectively deployed to the right audience.

In this challenge, we will be tackling the churn prediction problem on a very unique and interesting group of subscribers on a video streaming service!

Imagine that you are a new data scientist at this video streaming company and you are tasked with building a model that can predict which existing subscribers will continue their subscriptions for another month. We have provided a dataset that is a sample of subscriptions that were initiated in 2021, all snapshotted at a particular date before the subscription was cancelled. Subscription cancellation can happen for a multitude of reasons, including:
* the customer completes all content they were interested in, and no longer need the subscription
* the customer finds themselves to be too busy and cancels their subscription until a later time
* the customer determines that the streaming service is not the best fit for them, so they cancel and look for something better suited

Regardless the reason, this video streaming company has a vested interest in understanding the likelihood of each individual customer to churn in their subscription so that resources can be allocated appropriately to support customers. In this challenge, you will use your machine learning toolkit to do just that!

## Understanding the Datasets

### Train vs. Test
In this competition, you’ll gain access to two datasets that are samples of past subscriptions of a video streaming platform that contain information about the customer, the customers streaming preferences, and their activity in the subscription thus far. One dataset is titled `train.csv` and the other is titled `test.csv`.

`train.csv` contains 70% of the overall sample (243,787 subscriptions to be exact) and importantly, will reveal whether or not the subscription was continued into the next month (the “ground truth”).

The `test.csv` dataset contains the exact same information about the remaining segment of the overall sample (104,480 subscriptions to be exact), but does not disclose the “ground truth” for each subscription. It’s your job to predict this outcome!

Using the patterns you find in the `train.csv` data, predict whether the subscriptions in `test.csv` will be continued for another month, or not.

### Dataset descriptions
Both `train.csv` and `test.csv` contain one row for each unique subscription. For each subscription, a single observation (`CustomerID`) is included during which the subscription was active.

In addition to this identifier column, the `train.csv` dataset also contains the target label for the task, a binary column `Churn`.

Besides that column, both datasets have an identical set of features that can be used to train your model to make predictions. Below you can see descriptions of each feature. Familiarize yourself with them so that you can harness them most effectively for this machine learning task!

In [ ]:
import pandas as pd
# The original intent of this cell seemed to be to load data_descriptions.csv, which was not found.
# It then tried to load train.csv, which also failed.
# I'm commenting out these lines as they are redundant with the later data loading cells (c3S7APJpMV4V and sGBQesQPMV4W).
# pd.set_option('display.max_colwidth', None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

file_name = 'data_descriptions.csv'
file_found = False

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if file_name in files:
        print(f"'{file_name}' found at: {os.path.join(root, file_name)}")
        file_found = True
        break

if not file_found:
    print(f"'{file_name}' not found in your Google Drive.")

'data_descriptions.csv' not found in your Google Drive.


## How to Submit your Predictions to Coursera
Submission Format:

In this notebook you should follow the steps below to explore the data, train a model using the data in `train.csv`, and then score your model using the data in `test.csv`. Your final submission should be a dataframe (call it `prediction_df` with two columns and exactly 104,480 rows (plus a header row). The first column should be `CustomerID` so that we know which prediction belongs to which observation. The second column should be called `predicted_probability` and should be a numeric column representing the __likellihood that the subscription will churn__.

Your submission will show an error if you have extra columns (beyond `CustomerID` and `predicted_probability`) or extra rows. The order of the rows does not matter.

The naming convention of the dataframe and columns are critical for our autograding, so please make sure to use the exact naming conventions of `prediction_df` with column names `CustomerID` and `predicted_probability`!

To determine your final score, we will compare your `predicted_probability` predictions to the source of truth labels for the observations in `test.csv` and calculate the [ROC AUC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html). We choose this metric because we not only want to be able to predict which subscriptions will be retained, but also want a well-calibrated likelihood score that can be used to target interventions and support most accurately.

## Import Python Modules

First, import the primary modules that will be used in this project. Remember as this is an open-ended project please feel free to make use of any of your favorite libraries that you feel may be useful for this challenge. For example some of the following popular packages may be useful:

- pandas
- numpy
- Scipy
- Scikit-learn
- keras
- maplotlib
- seaborn
- etc, etc

In [ ]:
# Import required packages

# Data packages
import pandas as pd
import numpy as np

# Machine Learning / Classification packages
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

# Visualization Packages
from matplotlib import pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# Import any other packages you may want to use


## Load the Data

Let's start by loading the dataset `train.csv` into a dataframe `train_df`, and `test.csv` into a dataframe `test_df` and display the shape of the dataframes.

In [ ]:
try:
    train_df = pd.read_csv("/content/sample_data/train.csv")
    print('train_df Shape:', train_df.shape)
    display(train_df.head())
except FileNotFoundError:
    print("train.csv not found in /content/sample_data/. Please upload 'train.csv' or provide the correct path.")
    train_df = None # Set to None to avoid NameError if not loaded

train.csv not found in /content/sample_data/. Please upload 'train.csv' or provide the correct path.


In [ ]:
if train_df is not None:
    print('train_df was loaded successfully.')
    print('train_df Shape:', train_df.shape)
    display(train_df.head())
else:
    print('train_df was not loaded. Please ensure train.csv is in the correct path.')


train_df was not loaded. Please ensure train.csv is in the correct path.


In [ ]:
# Re-running the train_df loading cell to check if the file is now available
try:
    train_df = pd.read_csv('/content/sample_data/train.csv')
    print('train_df loaded successfully.')
    print('train_df Shape:', train_df.shape)
    display(train_df.head())
except FileNotFoundError:
    print("train.csv not found in /content/sample_data/. Please ensure 'train.csv' is uploaded.")
    train_df = None


train.csv not found in /content/sample_data/. Please ensure 'train.csv' is uploaded.


In [ ]:
try:
    train_df = pd.read_csv('/content/sample_data/train.csv')
    print('train.csv loaded successfully.')
    print('train_df Shape:', train_df.shape)
    display(train_df.head())
except FileNotFoundError:
    print("train.csv not found in /content/sample_data/. Please ensure 'train.csv' is uploaded.")
    train_df = None


train.csv not found in /content/sample_data/. Please ensure 'train.csv' is uploaded.


### Action Required: Upload `train.csv`

For the code above to execute successfully, please ensure you have uploaded `train.csv` to your Colab environment. The recommended location is `/content/sample_data/`.

You can upload files by:
1.  Clicking the folder icon on the left sidebar to open the File Browser.
2.  Navigating to the `/content/sample_data/` directory.
3.  Clicking the 'Upload' icon (a paperclip pointing upwards) and selecting your `train.csv` file.

After uploading, re-run the cell above and the subsequent cells that depend on `train_df` and `test_df` to resolve the `AttributeError`.

In [ ]:
# Re-running the test_df loading cell to check if the file is now available
try:
    test_df = pd.read_csv('/content/sample_data/test.csv')
    print('test_df loaded successfully.')
    print('test_df Shape:', test_df.shape)
    display(test_df.head())
except FileNotFoundError:
    print("test.csv not found in /content/sample_data/. Please ensure 'test.csv' is uploaded.")
    test_df = None


test.csv not found in /content/sample_data/. Please ensure 'test.csv' is uploaded.


In [ ]:
# Verify train_df status
if train_df is not None:
    print('train_df is available and loaded.')
    print('train_df Shape:', train_df.shape)
    display(train_df.head())
else:
    print('train_df is still not loaded.')

# Verify test_df status
if test_df is not None:
    print('test_df is available and loaded.')
    print('test_df Shape:', test_df.shape)
    display(test_df.head())
else:
    print('test_df is still not loaded.')


train_df is still not loaded.
test_df is still not loaded.


In [ ]:
import os

print('--- Files in /content/sample_data/ ---')
print(os.listdir('/content/sample_data/'))
print('\n')

print('--- Searching for train.csv and test.csv in Google Drive ---')
train_file_path = None
test_file_path = None

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train.csv' in files:
        train_file_path = os.path.join(root, 'train.csv')
        print(f"'train.csv' found at: {train_file_path}")
    if 'test.csv' in files:
        test_file_path = os.path.join(root, 'test.csv')
        print(f"'test.csv' found at: {test_file_path}")

if not train_file_path:
    print("'train.csv' not found in Google Drive.")
if not test_file_path:
    print("'test.csv' not found in Google Drive.")


# If files are found, update the loading cells accordingly
if train_file_path and train_file_path != '/content/sample_data/train.csv':
    print(f"\nAction required: Please update cell 'c3S7APJpMV4V' to load from: {train_file_path}")
elif not train_file_path:
    print("\nAction required: Please upload 'train.csv' to Colab or Google Drive.")

if test_file_path and test_file_path != '/content/sample_data/test.csv':
    print(f"Action required: Please update cell 'sGBQesQPMV4W' to load from: {test_file_path}")
elif not test_file_path:
    print("Action required: Please upload 'test.csv' to Colab or Google Drive.")


--- Files in /content/sample_data/ ---
['README.md', 'anscombe.json', 'california_housing_train.csv', 'california_housing_test.csv', 'mnist_train_small.csv', 'mnist_test.csv']


--- Searching for train.csv and test.csv in Google Drive ---
'train.csv' not found in Google Drive.
'test.csv' not found in Google Drive.

Action required: Please upload 'train.csv' to Colab or Google Drive.
Action required: Please upload 'test.csv' to Colab or Google Drive.


In [ ]:
try:
    test_df = pd.read_csv("/content/sample_data/test.csv")
    print('test_df Shape:', test_df.shape)
    display(test_df.head())
except FileNotFoundError:
    print("test.csv not found in /content/sample_data/. Please upload 'test.csv' or provide the correct path.")
    test_df = None # Set to None to avoid NameError if not loaded

test.csv not found in /content/sample_data/. Please upload 'test.csv' or provide the correct path.


## Explore, Clean, Validate, and Visualize the Data (optional)

Feel free to explore, clean, validate, and visualize the data however you see fit for this competition to help determine or optimize your predictive model. Please note - the final autograding will only be on the accuracy of the `prediction_df` predictions.

In [ ]:
# your code here (optional)

## Make predictions (required)

Remember you should create a dataframe named `prediction_df` with exactly 104,480 entries plus a header row attempting to predict the likelihood of retention for subscriptions in `test_df`. Your submission will throw an error if you have extra columns (beyond `CustomerID` and `predicted_probaility`) or extra rows.

The file should have exactly 2 columns:
`CustomerID` (sorted in any order)
`predicted_probability` (contains your numeric predicted probabilities between 0 and 1, e.g. from `estimator.predict_proba(X, y)[:, 1]`)

The naming convention of the dataframe and columns are critical for our autograding, so please make sure to use the exact naming conventions of `prediction_df` with column names `CustomerID` and `predicted_probability`!

### Example prediction submission:

The code below is a very naive prediction method that simply predicts retention using a Dummy Classifier. This is used as just an example showing the submission format required. Please change/alter/delete this code below and create your own improved prediction methods for generating `prediction_df`.

**PLEASE CHANGE CODE BELOW TO IMPLEMENT YOUR OWN PREDICTIONS**

In [ ]:
### PLEASE CHANGE THIS CODE TO IMPLEMENT YOUR OWN PREDICTIONS

# Fit a dummy classifier on the feature columns in train_df:
dummy_clf = DummyClassifier(strategy="stratified")
dummy_clf.fit(train_df.drop(['CustomerID', 'Churn'], axis=1), train_df.Churn)

AttributeError: 'NoneType' object has no attribute 'drop'

### Check for Missing Values

Before proceeding with data analysis, it's crucial to check for any missing values in your dataset. This can help you decide on appropriate data cleaning or imputation strategies.

**Note:** This step will only be effective once `train_df` has been successfully loaded from `train.csv`.

In [ ]:
if train_df is not None:
    print('Checking for missing values in train_df:')
    missing_values = train_df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
    if not missing_values.empty:
        display(missing_values.sort_values(ascending=False))
    else:
        print('No missing values found in train_df.')
else:
    print('train_df is not loaded. Please upload and load train.csv first.')


train_df is not loaded. Please upload and load train.csv first.


In [ ]:
### PLEASE CHANGE THIS CODE TO IMPLEMENT YOUR OWN PREDICTIONS

# Use our dummy classifier to make predictions on test_df using `predict_proba` method:
predicted_probability = dummy_clf.predict_proba(test_df.drop(['CustomerID'], axis=1))[:, 1]

In [ ]:
### PLEASE CHANGE THIS CODE TO IMPLEMENT YOUR OWN PREDICTIONS

# Combine predictions with label column into a dataframe
prediction_df = pd.DataFrame({'CustomerID': test_df[['CustomerID']].values[:, 0],
                             'predicted_probability': predicted_probability})

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import os

print('Listing files in /content/sample_data/ to verify uploads:')
current_files = os.listdir('/content/sample_data/')
print(current_files)

if 'train.csv' in current_files:
    print("'train.csv' is now present in /content/sample_data/.")
else:
    print("'train.csv' is still NOT present in /content/sample_data/.")

if 'test.csv' in current_files:
    print("'test.csv' is now present in /content/sample_data/.")
else:
    print("'test.csv' is still NOT present in /content/sample_data/.")

Listing files in /content/sample_data/ to verify uploads:
['README.md', 'anscombe.json', 'california_housing_train.csv', 'california_housing_test.csv', 'mnist_train_small.csv', 'mnist_test.csv']
'train.csv' is still NOT present in /content/sample_data/.
'test.csv' is still NOT present in /content/sample_data/.


In [ ]:
### PLEASE CHANGE THIS CODE TO IMPLEMENT YOUR OWN PREDICTIONS

# View our 'prediction_df' dataframe as required for submission.
# Ensure it should contain 104,480 rows and 2 columns 'CustomerID' and 'predicted_probaility'
print(prediction_df.shape)
prediction_df.head(10)

**PLEASE CHANGE CODE ABOVE TO IMPLEMENT YOUR OWN PREDICTIONS**

## Final Tests - **IMPORTANT** - the cells below must be run prior to submission

Below are some tests to ensure your submission is in the correct format for autograding. The autograding process accepts a csv `prediction_submission.csv` which we will generate from our `prediction_df` below. Please run the tests below an ensure no assertion errors are thrown.

In [ ]:
# FINAL TEST CELLS - please make sure all of your code is above these test cells

# Writing to csv for autograding purposes
prediction_df.to_csv("prediction_submission.csv", index=False)
submission = pd.read_csv("prediction_submission.csv")

assert isinstance(submission, pd.DataFrame), 'You should have a dataframe named prediction_df.'

In [ ]:
# FINAL TEST CELLS - please make sure all of your code is above these test cells

assert submission.columns[0] == 'CustomerID', 'The first column name should be CustomerID.'
assert submission.columns[1] == 'predicted_probability', 'The second column name should be predicted_probability.'

In [ ]:
# FINAL TEST CELLS - please make sure all of your code is above these test cells

assert submission.shape[0] == 104480, 'The dataframe prediction_df should have 104480 rows.'

In [ ]:
# FINAL TEST CELLS - please make sure all of your code is above these test cells

assert submission.shape[1] == 2, 'The dataframe prediction_df should have 2 columns.'

In [ ]:
# FINAL TEST CELLS - please make sure all of your code is above these test cells

## This cell calculates the auc score and is hidden. Submit Assignment to see AUC score.

## SUBMIT YOUR WORK!

Once we are happy with our `prediction_df` and `prediction_submission.csv` we can now submit for autograding! Submit by using the blue **Submit Assignment** at the top of your notebook. Don't worry if your initial submission isn't perfect as you have multiple submission attempts and will obtain some feedback after each submission!

In [ ]:
import os
print(os.listdir('.'))

In [ ]:
import os

print(os.listdir('.'))


In [ ]:
import os

print('Listing files in /content/:')
print(os.listdir('/content/'))


In [ ]:
import os

print('Listing files in /content/sample_data/:')
print(os.listdir('/content/sample_data/'))


In [ ]:
import pandas as pd

try:
    train_df = pd.read_csv('/content/sample_data/train.csv')
    print('train.csv loaded successfully.')
    print('First 5 rows of train_df:')
    display(train_df.head())
except FileNotFoundError:
    print("Error: 'train.csv' not found. Please ensure the file is uploaded to '/content/sample_data/'.")
    train_df = None


In [ ]:
import os

print('Files in the current directory:')
print(os.listdir('.'))


In [ ]:
import os

print('Files in /content/sample_data directory:')
print(os.listdir('/content/sample_data/'))


In [ ]:
import os

print('Listing files in /content/sample_data/ to check for uploaded files:')
current_files = os.listdir('/content/sample_data/')
print(current_files)

if 'train.csv' in current_files:
    print("'train.csv' is present in /content/sample_data/.")
else:
    print("'train.csv' is NOT yet present in /content/sample_data/.")

if 'test.csv' in current_files:
    print("'test.csv' is present in /content/sample_data/.")
else:
    print("'test.csv' is NOT yet present in /content/sample_data/.")
